<a href="https://colab.research.google.com/github/Areeha548/resnet-mnist-pytorch/blob/main/resnet_mnist_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# 1. Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Data
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("./data", train=True, download=True, transform=tf)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=256)

# 3. Model (ResNet-18, MNIST ke liye adjust)
model = models.resnet18(num_classes=10)
model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model = model.to(device)

# 4. Loss + Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 5. Training loop
for epoch in range(5):
    model.train()
    total_loss = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # 6. Testing
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    acc = 100 * correct / len(test_ds)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_dl):.4f} | Test Acc: {acc:.2f}%")

# 7. Save
torch.save(model.state_dict(), "resnet_mnist.pth")

Epoch 1 | Loss: 0.0996 | Test Acc: 97.77%
Epoch 2 | Loss: 0.0383 | Test Acc: 98.70%
Epoch 3 | Loss: 0.0279 | Test Acc: 99.24%
Epoch 4 | Loss: 0.0226 | Test Acc: 99.28%
Epoch 5 | Loss: 0.0183 | Test Acc: 99.21%
